# DraftGenerator 동작 방식 분석

**질문**: 현재 모델이 학습해서 결과를 도출한 건가요, RAG처럼 데이터를 검색해서 결과를 도출한 건가요?

**답변**: **둘 다 아닙니다.** 현재는 **Few-shot Prompting** 방식입니다.

## 1. 현재 구현 확인

In [ ]:
import sys
sys.path.insert(0, '..')

from src.analysis.draft_generator import FEW_SHOT_EXAMPLES

print(f"FEW_SHOT_EXAMPLES 타입: {type(FEW_SHOT_EXAMPLES)}")
print(f"FEW_SHOT_EXAMPLES 길이: {len(FEW_SHOT_EXAMPLES)} 문자")
print(f"\n=== 하드코딩된 예시 (처음 800자) ===")
print(FEW_SHOT_EXAMPLES[:800])

## 2. 세 가지 방식 비교

| 방식 | 설명 | 현재 구현 |
|------|------|----------|
| **Fine-tuning (학습)** | 모델 가중치를 수정하여 특정 도메인 학습 | ❌ 없음 |
| **RAG (검색 기반)** | 벡터 DB에서 유사 문서 검색 후 프롬프트에 포함 | ❌ 없음 |
| **Few-shot Prompting** | 하드코딩된 예시를 프롬프트에 포함 | ✅ **현재 방식** |

## 3. 현재 데이터 흐름 (Few-shot)

In [ ]:
# 현재 흐름 시각화
current_flow = """
┌─────────────────────────────────────────────────────────────────┐
│                    현재 구조 (Few-shot)                         │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  사용자 입력 (카카오톡 대화)                                    │
│       │                                                         │
│       ▼                                                         │
│  ┌─────────────────┐                                            │
│  │ EvidenceScorer  │  키워드 매칭으로 점수 산출                 │
│  │ (점수: 8.5/10)  │                                            │
│  └────────┬────────┘                                            │
│           │                                                     │
│           ▼                                                     │
│  ┌─────────────────┐   ┌─────────────────────────────────────┐  │
│  │ DraftGenerator  │◀──│ FEW_SHOT_EXAMPLES (하드코딩)        │  │
│  │                 │   │ - 제1호 예시 (부정행위)             │  │
│  │                 │   │ - 제2호 예시 (악의의 유기)          │  │
│  │                 │   │ - 제3호 예시 (부당한 대우)          │  │
│  │                 │   │ - 제6호 예시 (중대한 사유)          │  │
│  └────────┬────────┘   └─────────────────────────────────────┘  │
│           │                                                     │
│           ▼  GPT-4 API 호출 (system_prompt에 Few-shot 포함)     │
│  ┌─────────────────┐                                            │
│  │    GPT-4        │                                            │
│  │   (OpenAI)      │                                            │
│  └────────┬────────┘                                            │
│           │                                                     │
│           ▼                                                     │
│     이혼 소장 초안 생성                                         │
│                                                                 │
│  ❌ 벡터 DB 검색 없음                                           │
│  ❌ 동적 판례 검색 없음                                         │
│  ❌ 모델 학습/Fine-tuning 없음                                  │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
"""
print(current_flow)

## 4. 실제 코드 확인

In [ ]:
# draft_generator.py의 _generate_legal_reasoning 메서드 확인
import inspect
from src.analysis.draft_generator import DraftGenerator

# 소스 코드 확인
source = inspect.getsource(DraftGenerator._generate_legal_reasoning)
print("=== _generate_legal_reasoning 메서드 ===")
print(source[:2000])

## 5. 핵심 포인트

```python
# draft_generator.py 301행
system_prompt = f"""당신은 대한민국 가사법 전문 변호사입니다.
...
{FEW_SHOT_EXAMPLES}   # ← 여기서 하드코딩된 예시 사용
"""
```

**`FEW_SHOT_EXAMPLES`는 변수지만, 실제로는 코드에 하드코딩된 문자열입니다.**

- 벡터 DB에서 검색하지 않음
- 항상 동일한 4개 예시 사용
- 사건별 맞춤 판례 검색 불가

## 6. 이상적인 RAG 구조

In [ ]:
rag_flow = """
┌─────────────────────────────────────────────────────────────────┐
│                   이상적인 RAG 구조                             │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  사용자 입력 (카카오톡 대화)                                    │
│       │                                                         │
│       ▼                                                         │
│  ┌─────────────────┐                                            │
│  │ EvidenceScorer  │                                            │
│  └────────┬────────┘                                            │
│           │                                                     │
│           ▼                                                     │
│  ┌─────────────────┐   ┌─────────────────────────────────────┐  │
│  │ Query Generator │──▶│ Vector DB (Qdrant/OpenSearch)       │  │
│  │ "부정행위 외도" │   │ ┌─────────────────────────────────┐ │  │
│  └────────┬────────┘   │ │ 340개 민법 제840조 판례          │ │  │
│           │            │ │ - 각 판례 임베딩 벡터 저장       │ │  │
│           │ 유사도검색 │ │ - 메타데이터 (호수, 요지 등)     │ │  │
│           │◀───────────│ └─────────────────────────────────┘ │  │
│           │ Top-5 판례 └─────────────────────────────────────┘  │
│           ▼                                                     │
│  ┌─────────────────┐                                            │
│  │ GPT-4 + 검색된  │  검색된 유사 판례를 프롬프트에 동적 포함   │
│  │ 유사 판례       │                                            │
│  └────────┬────────┘                                            │
│           │                                                     │
│           ▼                                                     │
│     이혼 소장 초안 (실제 유사 판례 인용)                        │
│                                                                 │
│  ✅ 사건별 맞춤 판례 검색                                       │
│  ✅ 340개 전체 판례 활용 가능                                   │
│  ✅ 새 판례 추가 시 DB만 업데이트                               │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
"""
print(rag_flow)

## 7. 비교 요약

| 항목 | 현재 (Few-shot) | 이상적 (RAG) |
|------|-----------------|---------------|
| 판례 데이터 | 4개 예시 하드코딩 | 340개+ 벡터 DB |
| 유사 판례 검색 | ❌ 없음 | ✅ 임베딩 유사도 검색 |
| 사건별 맞춤 | ❌ 불가능 | ✅ Top-K 검색 |
| 새 판례 추가 | 코드 수정 필요 | DB 업데이트만 |
| 구현 복잡도 | 낮음 | 높음 |

## 8. 현재 저장된 판례 데이터 확인

In [ ]:
import os
import json

# 저장된 판례 데이터 확인
precedent_path = "../data/precedents/article840_samples.json"

if os.path.exists(precedent_path):
    with open(precedent_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    print(f"파일 크기: {len(content):,} bytes")
    print(f"\n처음 1000자:")
    print(content[:1000])
else:
    print("판례 데이터 파일이 없습니다.")

print("\n" + "="*60)
print("이 데이터는 저장만 되어 있고, 아직 RAG에 활용되지 않습니다!")
print("RAG 구현 시 이 데이터를 벡터 DB에 임베딩해야 합니다.")

## 9. 결론

### 현재 상태
- **학습(Fine-tuning)**: ❌ GPT-4 기본 모델 그대로 사용
- **RAG (검색 기반)**: ❌ 벡터 DB 검색 없음
- **Few-shot Prompting**: ✅ **현재 방식** - 하드코딩된 4개 예시 사용

### RAG 구현 시 필요한 것
1. **판례 임베딩**: 340개 판례 → OpenAI/Ollama 임베딩 → Qdrant 저장
2. **검색 모듈**: 쿼리 생성 → 유사도 검색 → Top-K 반환
3. **프롬프트 구성**: 검색된 판례를 동적으로 Few-shot 예시로 사용